# Week 3: Improve Model

Goal: improve the Week 2 CNN-LSTM image captioning baseline and compare it with an improved encoder-decoder model.

Main tasks:
- use ResNet50 or EfficientNet as encoder
- add dropout
- tune learning rate and batch size
- compare baseline vs improved model
- plot training/validation loss
- show good and bad caption examples


## 1. Imports and setup

This notebook continues the Flickr8k image captioning project. Run it after the Week 2 preprocessing notebook, or run all cells here from the beginning.

In [ ]:
# If needed in Colab:
# !pip install opendatasets nltk scikit-learn matplotlib

import os
import re
import string
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Load captions and images

In [ ]:
# Change paths if your folder names are different
text_path = "flickr8k/captions.txt"
images_path = "flickr8k/Images/"

df = pd.read_csv(text_path)
print("Total caption rows:", len(df))
print("Unique images:", df["image"].nunique())
df.head()

## 3. Clean captions and add tokens

In [ ]:
def clean_caption(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_caption"] = df["caption"].apply(clean_caption)
df["caption_tokens"] = df["clean_caption"].apply(lambda x: "<start> " + x + " <end>")

df[["image", "caption", "caption_tokens"]].head()

## 4. Train / validation / test split by image

We split by unique image names, not by caption rows. This prevents data leakage.

In [ ]:
unique_images = df["image"].unique()

train_imgs, temp_imgs = train_test_split(unique_images, test_size=0.20, random_state=SEED)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=SEED)

train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

print("Train images:", len(train_imgs), "captions:", len(train_df))
print("Val images:", len(val_imgs), "captions:", len(val_df))
print("Test images:", len(test_imgs), "captions:", len(test_df))

## 5. Build vocabulary from training captions

In [ ]:
MIN_FREQ = 5

counter = Counter()
for caption in train_df["caption_tokens"]:
    counter.update(caption.split())

special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
words = [word for word, count in counter.items() if count >= MIN_FREQ and word not in special_tokens]

itos = special_tokens + sorted(words)
stoi = {word: idx for idx, word in enumerate(itos)}

PAD_IDX = stoi["<pad>"]
START_IDX = stoi["<start>"]
END_IDX = stoi["<end>"]
UNK_IDX = stoi["<unk>"]

vocab_size = len(itos)
print("Vocabulary size:", vocab_size)

def encode_caption(caption):
    return [stoi.get(token, UNK_IDX) for token in caption.split()]

for split_df in [train_df, val_df, test_df]:
    split_df["encoded"] = split_df["caption_tokens"].apply(encode_caption)

train_df.head()

## 6. Extract image features with baseline and improved encoders

For comparison, this notebook supports:
- **Baseline:** VGG16 + LSTM
- **Improved:** ResNet50 + LSTM with dropout

If your Week 2 baseline used ResNet50 already, you can still use this table and write that Week 3 improved version has dropout + tuned hyperparameters.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def build_feature_extractor(encoder_name):
    encoder_name = encoder_name.lower()

    if encoder_name == "vgg16":
        weights = models.VGG16_Weights.DEFAULT
        model = models.vgg16(weights=weights)
        extractor = nn.Sequential(*list(model.classifier.children())[:-1])
        cnn = model.features
        avgpool = model.avgpool

        class VGGFeatureExtractor(nn.Module):
            def __init__(self, cnn, avgpool, classifier_part):
                super().__init__()
                self.cnn = cnn
                self.avgpool = avgpool
                self.classifier_part = classifier_part
            def forward(self, x):
                x = self.cnn(x)
                x = self.avgpool(x)
                x = torch.flatten(x, 1)
                x = self.classifier_part(x)
                return x

        feature_extractor = VGGFeatureExtractor(cnn, avgpool, extractor)
        feature_dim = 4096

    elif encoder_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        feature_extractor = nn.Sequential(*list(model.children())[:-1])
        feature_dim = 2048

    else:
        raise ValueError("encoder_name must be 'vgg16' or 'resnet50'")

    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()
    for p in feature_extractor.parameters():
        p.requires_grad = False

    return feature_extractor, feature_dim

def extract_features(image_names, encoder_name="resnet50", batch_size=64, save_dir="features"):
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"image_features_{encoder_name}.pt")

    if os.path.exists(save_path):
        print("Loading saved features:", save_path)
        return torch.load(save_path, map_location="cpu")

    feature_extractor, feature_dim = build_feature_extractor(encoder_name)
    features = {}

    image_names = list(image_names)
    for i in tqdm(range(0, len(image_names), batch_size)):
        batch_names = image_names[i:i + batch_size]
        images = []

        for image_name in batch_names:
            img_path = os.path.join(images_path, image_name)
            image = Image.open(img_path).convert("RGB")
            images.append(transform(image))

        images = torch.stack(images).to(device)

        with torch.no_grad():
            batch_features = feature_extractor(images)
            batch_features = batch_features.view(batch_features.size(0), -1).cpu()

        for image_name, feature in zip(batch_names, batch_features):
            features[image_name] = feature

    torch.save(features, save_path)
    print("Saved features:", save_path)
    print("Feature dimension:", feature_dim)
    return features

all_images = df["image"].unique()

# Baseline features: VGG16
vgg16_features = extract_features(all_images, encoder_name="vgg16", batch_size=32)

# Improved features: ResNet50
resnet50_features = extract_features(all_images, encoder_name="resnet50", batch_size=64)

## 7. Dataset and DataLoader

In [ ]:
class Flickr8kCaptionDataset(Dataset):
    def __init__(self, dataframe, image_features):
        self.df = dataframe.reset_index(drop=True)
        self.image_features = image_features

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = row["image"]
        feature = self.image_features[image_name]
        caption = torch.tensor(row["encoded"], dtype=torch.long)
        return feature, caption, image_name

def collate_fn(batch):
    features, captions, image_names = zip(*batch)
    features = torch.stack(features)

    max_len = max(len(c) for c in captions)
    padded = torch.full((len(captions), max_len), PAD_IDX, dtype=torch.long)

    for i, caption in enumerate(captions):
        padded[i, :len(caption)] = caption

    return features, padded, image_names

def make_loaders(image_features, batch_size):
    train_dataset = Flickr8kCaptionDataset(train_df, image_features)
    val_dataset = Flickr8kCaptionDataset(val_df, image_features)
    test_dataset = Flickr8kCaptionDataset(test_df, image_features)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader

## 8. CNN-LSTM model with optional dropout

The improved model uses dropout in the feature projection, embeddings, and LSTM output before the final classifier.

In [ ]:
class CNNLSTMCaptioner(nn.Module):
    def __init__(self, vocab_size, feature_dim, embed_dim=256, hidden_dim=512, pad_idx=0, dropout=0.0):
        super().__init__()
        self.feature_proj = nn.Linear(feature_dim, embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, captions):
        # teacher forcing: model sees all tokens except the last one
        caption_input = captions[:, :-1]

        img_embed = self.dropout(self.feature_proj(features)).unsqueeze(1)
        word_embed = self.dropout(self.embedding(caption_input))

        lstm_input = torch.cat([img_embed, word_embed], dim=1)
        output, _ = self.lstm(lstm_input)
        output = self.dropout(output)
        logits = self.fc(output)
        return logits

## 9. Training and validation functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0

    for features, captions, _ in tqdm(loader):
        features = features.to(device)
        captions = captions.to(device)

        logits = model(features, captions)
        targets = captions

        loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for features, captions, _ in loader:
            features = features.to(device)
            captions = captions.to(device)

            logits = model(features, captions)
            targets = captions
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            total_loss += loss.item()

    return total_loss / len(loader)

def train_model(model_name, image_features, feature_dim, batch_size, learning_rate, dropout, epochs=5):
    train_loader, val_loader, test_loader = make_loaders(image_features, batch_size=batch_size)

    model = CNNLSTMCaptioner(
        vocab_size=vocab_size,
        feature_dim=feature_dim,
        embed_dim=256,
        hidden_dim=512,
        pad_idx=PAD_IDX,
        dropout=dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = []
    best_val_loss = float("inf")
    best_path = f"{model_name.lower().replace(' ', '_')}_best.pt"

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss = validate_one_epoch(model, val_loader, criterion)

        history.append({
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "dropout": dropout
        })

        print(f"{model_name} | Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, pd.DataFrame(history), test_loader

## 10. Train baseline vs improved model

You can increase `EPOCHS` for better final results. Start with 3-5 epochs to check that everything works.

In [ ]:
EPOCHS = 5

# Baseline: VGG16 + LSTM, no dropout
baseline_model, baseline_history, baseline_test_loader = train_model(
    model_name="VGG16 + LSTM",
    image_features=vgg16_features,
    feature_dim=4096,
    batch_size=64,
    learning_rate=1e-3,
    dropout=0.0,
    epochs=EPOCHS
)

# Improved: ResNet50 + LSTM, dropout, tuned learning rate / batch size
improved_model, improved_history, improved_test_loader = train_model(
    model_name="ResNet50 + LSTM",
    image_features=resnet50_features,
    feature_dim=2048,
    batch_size=32,
    learning_rate=5e-4,
    dropout=0.3,
    epochs=EPOCHS
)

history_df = pd.concat([baseline_history, improved_history], ignore_index=True)
history_df

## 11. Plot training and validation loss

In [ ]:
plt.figure(figsize=(8, 5))
for model_name in history_df["model"].unique():
    subset = history_df[history_df["model"] == model_name]
    plt.plot(subset["epoch"], subset["train_loss"], marker="o", label=f"{model_name} train")
    plt.plot(subset["epoch"], subset["val_loss"], marker="o", linestyle="--", label=f"{model_name} val")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 12. Caption generation

In [ ]:
def generate_caption(model, feature, max_len=25):
    model.eval()

    generated = []
    input_word = torch.tensor([[START_IDX]], dtype=torch.long).to(device)
    feature = feature.unsqueeze(0).to(device)

    with torch.no_grad():
        img_embed = model.dropout(model.feature_proj(feature)).unsqueeze(1)
        output, hidden = model.lstm(img_embed)

        for _ in range(max_len):
            word_embed = model.embedding(input_word)
            output, hidden = model.lstm(word_embed, hidden)
            logits = model.fc(output.squeeze(1))
            predicted_idx = logits.argmax(dim=1).item()

            if predicted_idx == END_IDX:
                break

            predicted_word = itos[predicted_idx]
            if predicted_word not in ["<start>", "<pad>"]:
                generated.append(predicted_word)

            input_word = torch.tensor([[predicted_idx]], dtype=torch.long).to(device)

    return " ".join(generated)

## 13. BLEU evaluation

BLEU-1 checks unigram overlap. BLEU-2 checks unigram + bigram overlap.

In [ ]:
def evaluate_bleu(model, eval_df, image_features, max_images=None):
    references = []
    hypotheses = []

    image_names = list(eval_df["image"].unique())
    if max_images is not None:
        image_names = image_names[:max_images]

    for image_name in tqdm(image_names):
        refs = eval_df[eval_df["image"] == image_name]["clean_caption"].tolist()
        refs = [ref.split() for ref in refs]

        pred = generate_caption(model, image_features[image_name])
        hyp = pred.split()

        references.append(refs)
        hypotheses.append(hyp)

    bleu1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0))

    return {"BLEU-1": bleu1, "BLEU-2": bleu2}

baseline_bleu = evaluate_bleu(baseline_model, test_df, vgg16_features)
improved_bleu = evaluate_bleu(improved_model, test_df, resnet50_features)

comparison_df = pd.DataFrame([
    {"Model": "VGG16 + LSTM", "BLEU-1": baseline_bleu["BLEU-1"], "BLEU-2": baseline_bleu["BLEU-2"], "Notes": "baseline"},
    {"Model": "ResNet50 + LSTM", "BLEU-1": improved_bleu["BLEU-1"], "BLEU-2": improved_bleu["BLEU-2"], "Notes": "improved with dropout and tuned hyperparameters"}
])

comparison_df

## 14. Show good and bad caption examples

We score each generated caption using sentence BLEU-1 and then show the highest and lowest examples.

In [ ]:
def collect_caption_examples(model, eval_df, image_features, max_images=100):
    smooth = SmoothingFunction().method1
    examples = []

    image_names = list(eval_df["image"].unique())[:max_images]

    for image_name in tqdm(image_names):
        refs_text = eval_df[eval_df["image"] == image_name]["clean_caption"].tolist()
        refs_tokens = [r.split() for r in refs_text]
        pred = generate_caption(model, image_features[image_name])
        hyp_tokens = pred.split()

        score = sentence_bleu(refs_tokens, hyp_tokens, weights=(1.0, 0, 0, 0), smoothing_function=smooth)

        examples.append({
            "image": image_name,
            "prediction": pred,
            "reference_1": refs_text[0],
            "sentence_bleu_1": score
        })

    return pd.DataFrame(examples)

examples_df = collect_caption_examples(improved_model, test_df, resnet50_features, max_images=100)

good_examples = examples_df.sort_values("sentence_bleu_1", ascending=False).head(5)
bad_examples = examples_df.sort_values("sentence_bleu_1", ascending=True).head(5)

print("Good examples:")
display(good_examples)

print("Bad examples:")
display(bad_examples)

## 15. Visualize good and bad examples

In [ ]:
def show_examples(examples, title):
    for _, row in examples.iterrows():
        img_path = os.path.join(images_path, row["image"])
        image = Image.open(img_path).convert("RGB")

        plt.figure(figsize=(5, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.title(title)
        plt.show()

        print("Image:", row["image"])
        print("Prediction:", row["prediction"])
        print("Reference:", row["reference_1"])
        print("Sentence BLEU-1:", round(row["sentence_bleu_1"], 4))
        print("-" * 80)

show_examples(good_examples, "Good caption example")
show_examples(bad_examples, "Bad caption example")

## 16. Save Week 3 outputs

In [ ]:
os.makedirs("reports", exist_ok=True)

history_df.to_csv("reports/week3_training_history.csv", index=False)
comparison_df.to_csv("reports/week3_model_comparison.csv", index=False)
good_examples.to_csv("reports/week3_good_caption_examples.csv", index=False)
bad_examples.to_csv("reports/week3_bad_caption_examples.csv", index=False)

print("Saved:")
print("- reports/week3_training_history.csv")
print("- reports/week3_model_comparison.csv")
print("- reports/week3_good_caption_examples.csv")
print("- reports/week3_bad_caption_examples.csv")